In [0]:
%pip install google-search-results deltalake
dbutils.library.restartPython()

In [0]:
# Re-import packages after Python restart
import pandas as pd
import pyarrow as pa
from datetime import datetime
from serpapi import GoogleSearch
from deltalake.writer import write_deltalake
pd.set_option('display.max_columns', None)

In [0]:
MONITORING_CATALOG = "monitoring"
MONITORING_SCHEMA = "pipeline_ops"

In [0]:


# Set credentials and config - REPLACE WITH YOUR 
# Get from serpapi.com
FLIGHT_OUTBOUND = "2026-05-15"  
FLIGHT_RETURN = "2026-08-08"
Flight_Currency = "USD"
google_flights = "google_flights"
# ACTUAL SERPAPI KEY
SERPAPI_KEY = dbutils.secrets.get(scope="my-api-scope", key="serpapi-key") 

# Define schema for flight data using pyarrow
SCHEMA = pa.schema([
    ("flight_number", pa.string()),
    ("airline", pa.string()),
    ("origin", pa.string()),
    ("destination", pa.string()),
    ("departure_time", pa.string()),
    ("arrival_time", pa.string()),
    ("duration", pa.string()),
    ("stops", pa.int64()),
    ("price", pa.float64()),
    ("currency", pa.string()),
    ("trip_type", pa.string()),
    ("fetched_at", pa.string()),
])

DC_AIRPORTS = ["IAD","DCA","BWI"]
FL_AIRPORTS = ["MCO","MIA","FLL","TPA","RSW"]




In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, IntegerType, LongType, BooleanType
from datetime import datetime, timezone

def write_pipeline_metrics(pipeline_name, started_at, rows_processed,
    rows_quarantined, run_id, pipeline_version='1.0',
    slo_target_minutes=45, status='success', error_message=None
):
    # Import the catalog/schema variables from Cell 3
    from __main__ import MONITORING_CATALOG, MONITORING_SCHEMA
    
    # Define the metrics table name inside the function
    METRICS_TABLE = f"{MONITORING_CATALOG}.{MONITORING_SCHEMA}.etl_metrics"
    
    completed_at = datetime.now(timezone.utc)  # Changed from datetime.utcnow()
    duration_minutes = (completed_at - started_at).total_seconds() / 60
    slo_met = duration_minutes <= slo_target_minutes

    # Define the schema explicitly
    schema = StructType([
        StructField("run_id", StringType(), False),
        StructField("pipeline_name", StringType(), False),
        StructField("pipeline_version", StringType(), False),
        StructField("started_at", TimestampType(), False),
        StructField("completed_at", TimestampType(), False),
        StructField("duration_minutes", DoubleType(), False),
        StructField("rows_processed", LongType(), False),
        StructField("rows_quarantined", LongType(), False),
        StructField("slo_target_minutes", IntegerType(), False),
        StructField("slo_met", BooleanType(), False),
        StructField("status", StringType(), False),
        StructField("error_message", StringType(), True)  # Nullable
    ])

    row = Row(
        run_id=run_id,
        pipeline_name=pipeline_name,
        pipeline_version=pipeline_version,
        started_at=started_at,
        completed_at=completed_at,
        duration_minutes=round(duration_minutes, 2),
        rows_processed=rows_processed,
        rows_quarantined=rows_quarantined,
        slo_target_minutes=slo_target_minutes,
        slo_met=slo_met,
        status=status,
        error_message=error_message
    )

    # Create DataFrame with explicit schema
    (spark.createDataFrame([row], schema=schema)
        .write.format('delta')
        .mode('append')
        .saveAsTable(METRICS_TABLE))
    
    return duration_minutes, slo_met

In [0]:
def extract_flights(origin, destination):
    """Fetch flights for a single origin-destination pair"""
    params = {
        "engine": "google_flights",
        "departure_id": origin,
        "arrival_id": destination,
        "outbound_date": FLIGHT_OUTBOUND,
        "return_date": FLIGHT_RETURN,
        "currency": Flight_Currency,
        "hl": "en",
        "deep_search": "true",
        "api_key": SERPAPI_KEY,
    }

    search = GoogleSearch(params)
    results = search.get_dict()
    
    if "error" in results:
        print(f"[{origin} → {destination}] error: {results['error']}")
        return pd.DataFrame()

    flights = []
    fetched_at = datetime.utcnow().isoformat()

    for key in ["best_flights", "other_flights"]:
        for offer in results.get(key) or []:
            segs = offer.get("flights", [])
            if not segs:
                continue
            first = segs[0]
            last = segs[-1]
            flights.append({
                "flight_number": first.get("flight_number"),
                "airline": first.get("airline"),
                "origin": first.get("departure_airport", {}).get("id"),
                "destination": last.get("arrival_airport", {}).get("id"),
                "departure_time": first.get("departure_airport", {}).get("time"),
                "arrival_time": last.get("arrival_airport", {}).get("time"),
                "duration": str(offer.get("total_duration")),
                "stops": int(len(offer.get("layovers") or [])),
                "price": float(offer.get("price") or 0),
                "currency": Flight_Currency,
                "trip_type": "return",
                "fetched_at": fetched_at,
            })

    df = pd.DataFrame(flights)
    print(f"[{origin} → {destination}] flights fetched: {len(df)}")
    return df

def write_to_delta_table(df, table_name):
    """Writes pandas df to Delta table using Spark"""
    if df.empty:
        print(f"⚠️ Skipping {table_name} - no data")
        return
    
    # Convert pandas DataFrame to Spark DataFrame
    spark_df = spark.createDataFrame(df)
    
    # Write to Delta table (table_name must be quoted string)
    spark_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)
    
    print(f"✓ Wrote {len(df)} rows to {table_name}")



In [0]:
def extract_all_flights(origins, destinations):
    """Fetch flights for all origin-destination combinations"""
    all_flights = []
    
    for origin in FL_AIRPORTS:
        for destination in  DC_AIRPORTS:
            df = extract_flights(origin, destination)
            if not df.empty:
                all_flights.append(df)
    
    if all_flights:
       combined_df = pd.concat(all_flights, ignore_index=True)
       write_to_delta_table(combined_df, "workspace.bronze.Airflights_Return")
       return combined_df
    else:
        print("⚠️ No flights found")
        return pd.DataFrame()


import uuid
from datetime import timezone


# Fetch all DC → FL flights (3 x 5 = 15 combinations
start_time = datetime.now(timezone.utc)
run_id = str(uuid.uuid4())  # Generate unique run ID

# Fetch all DC → FL flights (3 x 5 = 15 combinations
df = extract_all_flights(FL_AIRPORTS, DC_AIRPORTS)

# Write pipeline metrics as the last step
write_pipeline_metrics(
    pipeline_name="AirFlight_SILVER",
    pipeline_version="1",
    started_at=start_time,
    rows_processed=len(df),
    rows_quarantined=0,
    run_id=run_id
)

In [0]:
%sql
Select* from monitoring.pipeline_ops.etl_metrics